# 魔改SHAP库
---

魔改SHAP库，以实现如下两个需求：
1. 联合Shapley：SHAP库默认是对每个点计算Shapley value，现想将多个点组合成super-pixel，并计算super-pixel的Shapley值；比如1024长度的振动信号，想通过组合降成5个super-pixel的分析；
2. Fix Shapley: 输入部分并非全部特征都需要计算shapley值（比如hilbert求模剩下的相位信息），因此想将一部分特征给固定下来，仅对剩下部分进行计算。

## SHAP库改动-联合Shapley：
`shap.maskers._tabular.py` 文件

**`masked_inputs_out` 针对 `np.dtype(object)` 的问题**:
```python
varying_rows_out = np.zeros((num_masks, self.shape[0]), dtype=bool)
masked_inputs_out = np.zeros((num_masks * self.shape[0], self.shape[1]), dtype=self.data.dtype) # tag #109
```

**`np.isclose`针对 `np.dtype(object)` 的问题**:
```python
if type(x.dtype) == np.dtype(object): # my addition # tag 153
    invariants = np.zeros(self.data.shape, dtype=bool)
    for i in range(self.data.shape[0]):
        for j in range(self.data.shape[1]):
            invariants[i, j] = np.all(np.isclose(x[j], self.data[i, j]))
    return invariants
else:
    return np.isclose(x, self.data)
```

**`_single_delta_mask`针对 `np.dtype(object)` 的问题**:
```python
def _single_delta_mask(dind, masked_inputs, last_mask, data, x, noop_code):
    if dind == noop_code:
        pass
    elif last_mask[dind]:
        for i in range(masked_inputs.shape[0]):  # tag 204
            masked_inputs[i, dind] = data[i, dind]
        # masked_inputs[:, dind] = data[:, dind]
        last_mask[dind] = False
    else:
        masked_inputs[:, dind] = x[dind:dind+1]  # tag 209
        # masked_inputs[:, dind] = x[dind]
        last_mask[dind] = True
```


**`_single_delta_mask`和 `_delta_masking` 关于 `numba` 的问题**:
```python
# @njit  # tag 197
def _single_delta_mask(dind, masked_inputs, last_mask, data, x, noop_code):
#---
# @njit  # tag 210
def _delta_masking(masks, x, curr_delta_inds, varying_rows_out,
                   masked_inputs_tmp, last_mask, data, variants, masked_inputs_out, noop_code):
```



## SHAP库改动-Fix Shapley：

### 改动方式1-统一data
`shap.maskers._tabular.py` 文件

**库文件改动**:
```python
class Tabular(Masker):
    def __init__(self, data, max_samples=100, clustering=None, fixed_background=None): # tag #19
#---
        self.data = data
        self.data_bak = data.copy() # tag 60
        self.fixed_background = fixed_background # tag #62
#---
    def __call__(self, mask, x):
        if self.fixed_background is not None:  # tag 96
            self.data = self.data_bak.copy()
            self.data[:,self.fixed_background] = x[self.fixed_background]
```

```python
class Independent(Tabular):
    def __init__(self, data, max_samples=100, fixed_background=None):
        super().__init__(data, max_samples=max_samples, clustering=None, fixed_background=fixed_background)
```

### 改动方式2-修改invariants
```python
class Tabular(Masker):
    def invariants(self, x):
    #...
        if type(x.dtype) == np.dtype(object):  # my addition # tag 153
            invariants = np.zeros(self.data.shape, dtype=bool)
            for i in range(self.data.shape[0]):
                for j in range(self.data.shape[1]):
                    invariants[i, j] = np.all(np.isclose(x[j], self.data[i, j]))
            result = invariants
        else:
            result = np.isclose(x, self.data)

        if self.fixed_background is not None: # my addition # tag 163
            result[:,self.fixed_background] = True
        return result
```

```python
class Independent(Tabular):
    def __init__(self, data, max_samples=100, fixed_background=None):
        super().__init__(data, max_samples=max_samples, clustering=None, fixed_background=fixed_background)
```
---

导入库

In [17]:
import shap
import numpy as np
import sklearn

简单地训练下模型

In [18]:
X_adult,y_adult = shap.datasets.adult()
model_adult = sklearn.linear_model.LogisticRegression(max_iter=10000)
model_adult.fit(X_adult.values, y_adult)

LogisticRegression(max_iter=10000)

**未魔改的情况(V1)**:


In [19]:
def model_adult_probaV1(x):
    x = [np.hstack(item) for item in x]
    return model_adult.predict_proba(x)[:,1]
tempV1 = X_adult.values

background_adultV1 = shap.maskers.Independent(tempV1[:100], max_samples=10)
explainerV1 = shap.Explainer(model_adult_probaV1, background_adultV1, algorithm='exact')
shap_values_adultV1 = explainerV1(tempV1[:5])

In [20]:
print(shap_values_adultV1.values)
print(f'shape:{str(shap_values_adultV1.values.shape):s}')
delta,base = shap_values_adultV1.values.sum(1), shap_values_adultV1.base_values
result = base + delta
predict = model_adult_probaV1(tempV1[:5])
print(f' {"base:":>8s}{str(base):s}\n {"delta:":>8s}{str(delta):s}\n')
print(f' {"predict:":>8s}{str(predict):s}\n {"result:":>8s}{str(result):s}')

[[ 0.01354413 -0.00956282  0.13841467 -0.03055662 -0.01158353 -0.15529991
   0.00428134  0.00753143  0.07235438 -0.01503305 -0.00595312  0.00335446]
 [ 0.07982952 -0.00960944  0.20579658  0.01337965 -0.00957382  0.11904227
   0.00581157  0.01023517  0.         -0.01999424 -0.14853137  0.00462821]
 [ 0.00481566 -0.00044103 -0.00685234  0.02963312 -0.003036   -0.10219447
   0.003571    0.00633101  0.         -0.01194503 -0.0023387   0.00263502]
 [ 0.0644798  -0.00062083 -0.09466424  0.00752039 -0.00397255  0.07710225
  -0.01801889  0.00816253  0.         -0.01634449 -0.00677379  0.0035578 ]
 [-0.03257784 -0.00070768  0.20568434  0.0135578   0.00442454  0.19854305
  -0.02539599 -0.07217438  0.         -0.02059295 -0.00858797 -0.02141124]]
shape:(5, 12)
    base:[0.11886367 0.11886367 0.11886367 0.11886367 0.11886367]
   delta:[ 0.01149138  0.2510141  -0.07982174  0.020428    0.24076169]

 predict:[0.13035505 0.36987777 0.03904193 0.13929167 0.35962536]
  result:[0.13035505 0.36987777 0.03

**魔改的情况-联合(V2)**:

In [21]:
def model_adult_probaV2(x):
    x = [np.hstack(item) for item in x]
    return model_adult.predict_proba(x)[:,1]

tempV2 = [[item[:3],item[3:9],item[9:]] for item in X_adult.values]
tempV2 = np.array(tempV2,dtype=object)

background_adultV2 = shap.maskers.Independent(tempV2[:10], max_samples=10)
explainerV2 = shap.Explainer(model_adult_probaV2, background_adultV2, algorithm='exact')
shap_values_adultV2 = explainerV2(tempV2[:5])

In [22]:
print(shap_values_adultV2.values)
print(f'shape:{str(shap_values_adultV2.values.shape):s}')
delta,base = shap_values_adultV2.values.sum(1), shap_values_adultV2.base_values
result = base + delta
predict = model_adult_probaV2(tempV2[:5])
print(f' {"base:":>8s}{str(base):s}\n {"delta:":>8s}{str(delta):s}\n')
print(f' {"predict:":>8s}{str(predict):s}\n {"result:":>8s}{str(result):s}')

[[ 0.03879357 -0.29190902  0.01425065]
 [ 0.13561599  0.01190993 -0.146868  ]
 [-0.11864238 -0.22066847  0.00913294]
 [-0.20255546 -0.04065291  0.01328019]
 [ 0.01266733 -0.01226863 -0.00999319]]
shape:(5, 3)
    base:[0.36921985 0.36921985 0.36921985 0.36921985 0.36921985]
   delta:[-0.2388648   0.00065793 -0.33017791 -0.22992817 -0.00959449]

 predict:[0.13035505 0.36987777 0.03904193 0.13929167 0.35962536]
  result:[0.13035505 0.36987777 0.03904193 0.13929167 0.35962536]


**魔改的情况-联合+Fix(V3)**:

In [23]:
# 通过强行修改invariants来实现，base值不变，但base + delta != predict
def model_adult_probaV3(x):
    x = [np.hstack(item) for item in x]
    return model_adult.predict_proba(x)[:,1]

tempV3 = [[item[:3],item[3:9],item[9:]] for item in X_adult.values]
tempV3 = np.array(tempV3,dtype=object)

background_adultV3 = shap.maskers.Independent(tempV3[:10], max_samples=10, fixed_background=[False,False,True])
explainerV3 = shap.Explainer(model_adult_probaV3, background_adultV3, algorithm='permutation')
shap_values_adultV3 = explainerV3(tempV3[:5])

In [24]:
print(shap_values_adultV3.values)
print(f'shape:{str(shap_values_adultV3.values.shape):s}')
delta,base = shap_values_adultV3.values.sum(1), shap_values_adultV3.base_values
result = base + delta
predict = model_adult_probaV3(tempV3[:5])
print(f' {"base:":>8s}{str(base):s}\n {"delta:":>8s}{str(delta):s}\n')
print(f' {"predict:":>8s}{str(predict):s}\n {"result:":>8s}{str(result):s}')

[[ 0.03755414 -0.28649141  0.        ]
 [ 0.1543899   0.03529398  0.        ]
 [-0.11447853 -0.21868827  0.        ]
 [-0.19689412 -0.04379912  0.        ]
 [ 0.0127932  -0.01133374  0.        ]]
shape:(5, 3)
    base:[0.36921985 0.36921985 0.36921985 0.36921985 0.36921985]
   delta:[-0.24893727  0.18968387 -0.33316679 -0.24069324  0.00145946]

 predict:[0.13035505 0.36987777 0.03904193 0.13929167 0.35962536]
  result:[0.12028258 0.55890372 0.03605305 0.12852661 0.37067931]
